# Predictive Maintenance: Machine Failure Classification

This notebook predicts whether a machine will fail based on sensor readings, using the AI4I 2020 Predictive Maintenance dataset (10,000 rows). It compares three models — Logistic Regression, Random Forest, and XGBoost — and selects a final model based on precision/recall trade-offs rather than raw accuracy, since failures are rare (~3.4% of records).

**Workflow:** Data understanding → EDA → feature engineering → model comparison → threshold tuning → feature importance.

## 1. Imports

In [2]:
import pandas as pd


## 2. Load data

In [3]:
df = pd.read_csv("maintainance.csv")

# Clean column names: some ML libraries (e.g. XGBoost) reject "[" "]" characters
df.columns = df.columns.str.replace(r"\s*\[.*?\]", "", regex=True).str.strip()


## 3. Understand & clean the data

Check shape, data types, missing values, and — most importantly — the class balance of the target variable, since this shapes every decision later in the project.

In [4]:
df.shape


(10000, 14)

In [5]:
df.dtypes.value_counts()


,count
int64,9
float64,3
object,2


In [6]:
df.isnull().sum().sum()


np.int64(0)

In [7]:
print(df['Machine failure'].value_counts())
print(df['Machine failure'].value_counts(normalize=True) * 100)


Machine failure
0    9661
1     339
Name: count, dtype: int64
Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64


**Finding:** only 3.39% of records are failures (339 out of 10,000). This severe class imbalance means accuracy alone is a misleading metric — precision and recall on the failure class matter far more.

In [8]:
df = df.drop(columns=['UDI', 'Product ID'])  # identifiers, no predictive value
df.head()


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 4. Exploratory data analysis

Three targeted checks instead of a plot per feature: do sensor readings differ between failed and non-failed machines, which failure type is most common, and does product quality tier (Type) affect failure rate.

In [9]:
# Average sensor values: failure vs no failure
df.groupby('Machine failure')[['Air temperature', 'Process temperature',
                                 'Rotational speed', 'Torque',
                                 'Tool wear']].mean().round(2)


,Air temperature,Process temperature,Rotational speed,Torque,Tool wear
Machine failure,,,,,
0,299.97,310.00,1540.26,39.63,106.69
1,300.89,310.29,1496.49,50.17,143.78


In [10]:
# Which failure type is most common among the 339 failures
df[['TWF','HDF','PWF','OSF','RNF']].sum()


,0
TWF,46
HDF,115
PWF,95
OSF,98
RNF,19


In [11]:
# Does product Type (L/M/H) affect failure rate?
df.groupby('Type')['Machine failure'].mean() * 100


,Machine failure
Type,
H,2.093719
L,3.916667
M,2.769436


**Findings:** Torque and Tool wear show the clearest separation between failed and non-failed machines. Heat Dissipation Failure (HDF) is the most common failure type. Lower-quality product tier (Type L) fails almost twice as often as high-quality (Type H).

## 5. Feature engineering

**Important — data leakage check:** the columns `TWF`, `HDF`, `PWF`, `OSF`, `RNF` record *which* failure mechanism occurred — they are outcome labels generated alongside `Machine failure`, not independent sensor inputs. Including them as features would let the model "cheat" by learning a near-identity relationship, producing a model that looks accurate but is useless in a real deployment where these labels aren't known in advance. They are dropped from the feature set here and reserved as the target for a future multi-class (failure-type) model.

In [12]:
# One-hot encode Type (L/M/H); convert resulting booleans to 0/1 integers
df = pd.get_dummies(df, columns=['Type'], drop_first=False)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
df.head()


,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF,Type_H,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,0,0,1
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,0,1,0
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,0,1,0
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,0,1,0
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,0,1,0


In [13]:
# Features (X) and target (y) — leakage columns excluded from X
X = df.drop(columns=['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])
y = df['Machine failure']

print(X.columns.tolist())
print(X.shape, y.shape)


['Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Type_H', 'Type_L', 'Type_M']
(10000, 8) (10000,)


## 6. Train/test split and scaling

Stratified split preserves the 96.6% / 3.4% class ratio in both train and test sets. Scaling is fit only on training data and applied to test data — fitting on the full dataset before splitting would leak test-set information into training.

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaled versions — used only for Logistic Regression (tree models don't need scaling)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)


## 7. Baseline model — Logistic Regression

`class_weight='balanced'` compensates for the rare failure class by weighting it more heavily during training.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))


Accuracy: 0.825

Confusion Matrix:
[[1594  338]
 [  12   56]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.83      0.90      1932
           1       0.14      0.82      0.24        68

    accuracy                           0.82      2000
   macro avg       0.57      0.82      0.57      2000
weighted avg       0.96      0.82      0.88      2000



**Result:** high recall (0.82) on the failure class, but precision collapses to 0.14 — 338 false alarms out of 2,000 test machines. Useful as a baseline, but too many false alarms for practical use.

## 8. Random Forest

Tree-based models capture non-linear feature interactions and don't require scaled input. Class weights are set manually to bias the model toward catching more failures.

In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=250, random_state=42, class_weight={0: 1, 1: 10})
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_pred))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))
print()
print("Classification Report:")
print(classification_report(y_test, rf_pred))


Accuracy: 0.9815

Confusion Matrix:
[[1929    3]
 [  34   34]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.92      0.50      0.65        68

    accuracy                           0.98      2000
   macro avg       0.95      0.75      0.82      2000
weighted avg       0.98      0.98      0.98      2000



**Result:** much better precision (0.92) than Logistic Regression, but lower recall (0.49) — it misses more failures than we'd like, despite being very trustworthy when it does flag one.

## 9. XGBoost — final model

Gradient boosting builds trees sequentially, each correcting the previous tree's errors — well suited to a hard minority class. `scale_pos_weight` (ratio of non-failures to failures) replaces `class_weight` for imbalance handling. A custom decision threshold of 0.7 (tuned by testing several values) is applied to the predicted probabilities instead of the default 0.5, improving both precision and recall simultaneously over the default threshold.

In [17]:
from xgboost import XGBClassifier

scale = (y_train == 0).sum() / (y_train == 1).sum()
print("Scale pos weight:", scale)

xgb_model = XGBClassifier(random_state=42, scale_pos_weight=scale, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Custom threshold (0.7) instead of default 0.5 — tuned by testing a range of values
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_pred = (xgb_probs > 0.7).astype(int)

print("Accuracy:", accuracy_score(y_test, xgb_pred))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))
print()
print("Classification Report:")
print(classification_report(y_test, xgb_pred, target_names=['No Failure', 'Failure']))


Scale pos weight: 28.52029520295203
Accuracy: 0.9855

Confusion Matrix:
[[1919   13]
 [  16   52]]

Classification Report:
              precision    recall  f1-score   support

  No Failure       0.99      0.99      0.99      1932
     Failure       0.80      0.76      0.78        68

    accuracy                           0.99      2000
   macro avg       0.90      0.88      0.89      2000
weighted avg       0.99      0.99      0.99      2000



**Result: best model overall.** Catches 76% of real failures (52/68) while raising only 13 false alarms — the strongest precision/recall balance (F1 = 0.78) of every model and configuration tested.

## 10. Feature importance

Which features did the final XGBoost model actually rely on?

In [17]:
importances = xgb_model.feature_importances_
feature_names = X_train.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

importance_df


,Feature,Importance
2,Rotational speed,0.330352
3,Torque,0.250173
4,Tool wear,0.219431
0,Air temperature,0.057252
7,Type_M,0.045142
6,Type_L,0.043271
1,Process temperature,0.040169
5,Type_H,0.014210


**Finding:** Rotational speed, Torque, and Tool wear dominate (together ~80% of total importance). Rotational speed ranks highest despite showing only a small average difference in EDA — likely because it matters through *interaction* with other features (e.g. high speed combined with high torque), a pattern only a model that splits on feature combinations, not simple averages, can detect. Product Type contributes little once actual sensor readings are available, since Type was only ever a rough proxy for the machine's real operating conditions.

## 11. Summary

| Model | Accuracy | Precision (failure) | Recall (failure) | F1 (failure) |
|---|---|---|---|---|
| Logistic Regression (balanced) | 82.5% | 0.14 | 0.82 | 0.24 |
| Random Forest (weighted) | 98.1% | 0.92 | 0.49 | 0.63 |
| **XGBoost (threshold 0.7)** | **99%** | **0.80** | **0.76** | **0.78** |

**Final model:** XGBoost with `scale_pos_weight` for class imbalance and a tuned decision threshold of 0.7, selected for the best balance of precision and recall on the failure class — the metric that matters most for a predictive maintenance use case, where both missed failures and false alarms carry real costs.

**Limitations:** review/informal analysis aside, this model is trained on 10,000 records with only 339 failures; a larger real-world dataset would give more reliable estimates. Random Failures (RNF) are unpredictable by design and represent a known ceiling on achievable recall.

**Next step:** a multi-class model predicting *which* failure type occurs (TWF/HDF/PWF/OSF/RNF), using the same cleaned feature set.

In [18]:
# Count how many failure flags are set to 1 in each row
df['num_failures_flagged'] = df[['TWF','HDF','PWF','OSF','RNF']].sum(axis=1)
df['num_failures_flagged'].value_counts()

,count
num_failures_flagged,
0,9652
1,324
2,23
3,1


In [19]:
# Keep only rows with 0 or 1 failure flags set - drop genuinely ambiguous rows
df_clean = df[df['num_failures_flagged'] <= 1].copy()
print(df_clean.shape)
print(df_clean['num_failures_flagged'].value_counts())

(9976, 15)
num_failures_flagged
0    9652
1     324
Name: count, dtype: int64


In [20]:
# Original code to create the Failure_Type_Label column
failure_columns = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

def get_failure_labels(row):
    labels = [col for col in failure_columns if row[col] == 1]
    if labels:
        return labels[0]
    else:
        return 'No Failure'

df_clean['Failure_Type_Label'] = df.apply(get_failure_labels, axis=1)

# Display the new column along with the original failure columns to verify
print(df_clean[['TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Failure_Type_Label']].head())

   TWF  HDF  PWF  OSF  RNF Failure_Type_Label
0    0    0    0    0    0         No Failure
1    0    0    0    0    0         No Failure
2    0    0    0    0    0         No Failure
3    0    0    0    0    0         No Failure
4    0    0    0    0    0         No Failure


In [21]:
df_clean['Failure_Type_Label'].value_counts()

,count
Failure_Type_Label,
No Failure,9652
HDF,106
PWF,80
OSF,78
TWF,42
RNF,18


In [22]:
X= df_clean.drop(columns=['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Failure_Type_Label','num_failures_flagged'])
y= df_clean['Failure_Type_Label']

In [23]:
X.columns = X.columns.str.replace(r"\s*\[.*?\]", "", regex=True).str.strip()
X_train.columns = X_train.columns.str.replace(r"\s*\[.*?\]", "", regex=True).str.strip()
X_test.columns = X_test.columns.str.replace(r"\s*\[.*?\]", "", regex=True).str.strip()

print(X_train.columns.tolist())  # confirm no more brackets

['Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Type_H', 'Type_L', 'Type_M']


In [24]:
X.shape

(9976, 8)

In [25]:
X.head()

,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Type_H,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,0,1
1,298.2,308.7,1408,46.3,3,0,1,0
2,298.1,308.5,1498,49.4,5,0,1,0
3,298.2,308.6,1433,39.5,7,0,1,0
4,298.2,308.7,1408,40.0,9,0,1,0


In [26]:
y.shape

(9976,)

In [27]:
y.value_counts()

,count
Failure_Type_Label,
No Failure,9652
HDF,106
PWF,80
OSF,78
TWF,42
RNF,18


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(7980, 8) (1996, 8)
Failure_Type_Label
No Failure    7721
HDF             85
PWF             64
OSF             62
TWF             34
RNF             14
Name: count, dtype: int64
Failure_Type_Label
No Failure    1931
HDF             21
PWF             16
OSF             16
TWF              8
RNF              4
Name: count, dtype: int64


In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

rf_multi = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_multi.fit(X_train, y_train)

y_pred = rf_multi.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9804609218436874

Confusion Matrix:
[[  14    7    0    0    0    0]
 [   0 1930    0    1    0    0]
 [   0    7    9    0    0    0]
 [   0   12    0    4    0    0]
 [   0    4    0    0    0    0]
 [   0    8    0    0    0    0]]

Classification Report:
              precision    recall  f1-score   support

         HDF       1.00      0.67      0.80        21
  No Failure       0.98      1.00      0.99      1931
         OSF       1.00      0.56      0.72        16
         PWF       0.80      0.25      0.38        16
         RNF       0.00      0.00      0.00         4
         TWF       0.00      0.00      0.00         8

    accuracy                           0.98      1996
   macro avg       0.63      0.41      0.48      1996
weighted avg       0.97      0.98      0.98      1996



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [30]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Encode string labels (HDF, PWF, etc.) into numbers - XGBoost's core expects numeric classes
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Check the mapping so you can translate results back later
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'HDF': np.int64(0), 'No Failure': np.int64(1), 'OSF': np.int64(2), 'PWF': np.int64(3), 'RNF': np.int64(4), 'TWF': np.int64(5)}


In [31]:
xgb_multi = XGBClassifier(random_state=42, objective='multi:softmax', num_class=len(le.classes_), eval_metric='mlogloss')
xgb_multi.fit(X_train, y_train_enc)

y_pred_enc = xgb_multi.predict(X_test)

# Decode predictions back to original labels for a readable report
y_pred = le.inverse_transform(y_pred_enc)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9834669338677354

Confusion Matrix:
[[  18    3    0    0    0    0]
 [   1 1923    2    2    1    2]
 [   0    4   12    0    0    0]
 [   0    6    0   10    0    0]
 [   0    4    0    0    0    0]
 [   0    8    0    0    0    0]]

Classification Report:
              precision    recall  f1-score   support

         HDF       0.95      0.86      0.90        21
  No Failure       0.99      1.00      0.99      1931
         OSF       0.86      0.75      0.80        16
         PWF       0.83      0.62      0.71        16
         RNF       0.00      0.00      0.00         4
         TWF       0.00      0.00      0.00         8

    accuracy                           0.98      1996
   macro avg       0.60      0.54      0.57      1996
weighted avg       0.98      0.98      0.98      1996



In [32]:
# In Colab, instead of joblib.dump for the xgboost model specifically:
xgb_model.save_model('binary_failure_model.json')

In [33]:
# In Colab, instead of joblib.dump for the xgboost model specifically:
xgb_multi.save_model('failure_type_model.json')


In [34]:
import joblib
joblib.dump(le, 'label_encoder.pkl')

['label_encoder.pkl']